In [5]:
pip install plotly

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/9.7 MB 16.8 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 17.5 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 17.8 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 17.8 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 17.8 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 17.8 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 7.5 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm
from matplotlib import pyplot as plt
import plotly.graph_objects as go

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X should have shape (n_samples, 2)
# y should have shape (n_samples,)
# ==========================================

# Example
X = np.load('function_1/initial_inputs.npy')
y = np.load('function_1/initial_outputs.npy')
X = np.vstack([X, [0.776414, 0.503573], 
               [0.807341, 0.333940],
               [0.729278, 0.453659],
               [0.527266, 0.206204],
               [0.201714, 0.543530],
               [0.740095, 0.811547],
               [0.324594, 0.625493],
               [0.237314, 0.368713],
               [0.872321, 0.112901],
               [0.175202, 0.299072],
               [0.368634, 0.177062]])

y = np.append(y, [4.7375798129026724e-27, 
                  2.7523543783915092e-83,
                  -4.753882395112984e-29,
                  3.34978523495855e-41,
                  -1.6854734804873104e-45,
                  -3.0303239460452828e-33,
                  8.030375202598981e-37,
                  -7.67894983993911e-27,
                  2.6127538278137392e-208,
                  -4.7017274366237273e-54,
                  1.7428722941557693e-44])

# Log of y (base 10)
log_y = np.log10(y)

# Create the 3D scatter plot
# Plot all points except the last one
# Create figure
fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=X[:-1, 0],
        y=X[:-1, 1],
        z=log_y[:-1],
        mode="markers",
        marker=dict(
            size=4,
            color="blue",
            opacity=0.8,
        ),
        name="Data points",
    )
)

# Plot the last point with a different color
fig.add_trace(
    go.Scatter3d(
        x=[X[-1, 0]],
        y=[X[-1, 1]],
        z=[log_y[-1]],
        mode="markers",
        marker=dict(
            size=8,
            color="red",
            symbol="diamond",
        ),
        name="Last point",
    )
)

# Layout
fig.update_layout(
    title="3D Scatter Plot with Last Point Highlighted",
    scene=dict(
        xaxis_title="X1",
        yaxis_title="X2",
        zaxis_title="log10(Y)",
    ),
    template="plotly_white",
)

fig.show()

C:\Users\mho2\AppData\Local\Temp\ipykernel_23784\2341241998.py:35: RuntimeWarning: invalid value encountered in log10
  log_y = np.log10(y)


In [ ]:
#In a Matérn kernel for a Gaussian Process Regressor, nu (ν) controls the smoothness (roughness) of the learned function, and length_scale (l) controls how far the influence of a data point reaches across the input space
#Small values (like ν = 0.5 or 1.5) create rough, sharp, or jagged curves
#Large values (like ν = 2.5) create smooth, flowing curves. As ν → ∞, it becomes the standard RBF (Squared Exponential) kernel
#Common default choices in machine learning are 1.5 and 2.5

In [4]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=[1.0, 1.0], nu=1.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# 3. Create a dense grid to map the continuous 3D surface
bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits
bounds = np.array(bounds)
x_line = np.linspace(bounds[0, 0], bounds[0, 1], 100)
y_line = np.linspace(bounds[1, 0], bounds[1, 1], 100)
X_grid, Y_grid = np.meshgrid(x_line, y_line)

grid_points = np.vstack([X_grid.ravel(), Y_grid.ravel()]).T

# Predict scores across the entire dense grid
Z_grid = gp.predict(grid_points).reshape(X_grid.shape)
log_Z_grid = np.log10(Z_grid)

# 4. Define the Classification Threshold (e.g., top 20% of outputs are "Good")
threshold = np.percentile(y, 80)
# Create a flat 3D plane reflecting our decision boundary
Z_boundary = np.full_like(Z_grid, fill_value=threshold)

# 5. Construct the Interactive Plotly 3D Figure
fig = go.Figure()

# Trace 1: The Predicted Continuous Response Surface
fig.add_trace(go.Surface(
    x=x_line, y=y_line, z=log_Z_grid,
    colorscale='Viridis',
    name='Predicted Landscape',
    colorbar=dict(title='Score Value')
))

# Trace 2: The Semi-Transparent Decision Boundary Plane
fig.add_trace(go.Surface(
    x=x_line, y=y_line, z=np.log10(Z_boundary),
    colorscale='Reds',
    opacity=0.4,
    showscale=False,
    name='Good/Bad Threshold'
))

# Trace 3: Real Evaluated Points (Scatter)
fig.add_trace(go.Scatter3d(
    x=X[:, 0], y=X[:, 1], z=log_y,
    mode='markers',
    marker=dict(size=5, color='orange', symbol='circle'),
    name='Submitted Queries'
))

# Layout configurations for crisp visualization
fig.update_layout(
    title='BBO 3D Surface Plot with Classification Decision Boundary',
    scene=dict(
        xaxis_title='Feature 1',
        yaxis_title='Feature 2',
        zaxis_title='Log Output Score'
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

# Display the interactive window
fig.show()

C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\mho2\AppData\Local\Temp\ipykernel_123064\3360736057.py:31: RuntimeWarning: invalid value encountered in log10
  log_Z_grid = np.log10(Z_grid)


In [4]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=[1.0, 1.0], nu=1.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# xi: The exploration-exploitation trade-off parameter (default 0.01), where higher values encourage searching in uncertain areas.
#0.01 (Standard): Delivers a balanced mix of exploration and exploitation.
#0.1 or higher: Forces the algorithm to explore high-uncertainty areas.
#0.001 or lower: Forces the algorithm to exploit known good regions.

#Large xi: Ignores small improvements. Seeks areas with massive potential variance.
#Small xi: Targets any slight improvement. Locks onto local minima/maxima quickly.
#Dynamic Schedules: Many advanced systems start with a high xi (e.g., 0.1) and decay it toward 0 over time.
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.001):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(10000):

    # Randomly sample a point within the bounds
    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Next point to evaluate:
[0.103611 0.235586]
Expected Improvement:
8.170308728153209e-06
